<a href="https://colab.research.google.com/github/dhurga-13/phishing_detection/blob/main/phishing_website.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Environment Setup and Library Installation
!pip install python-whois tld

import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Configure Kaggle API
# Please upload your kaggle.json file to the Colab environment before running this cell.
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("Environment setup complete. Necessary libraries are installed and Kaggle API is configured.")

In [ ]:
# Cell 2: Data Collection and Initial Exploration

# Download the dataset from Kaggle
!kaggle datasets download -d shashwatwork/web-page-phishing-detection-dataset

# Unzip the dataset
!unzip -o web-page-phishing-detection-dataset.zip

# Load the dataset into a pandas DataFrame
try:
    df = pd.read_csv('dataset_phishing.csv')
    print("Dataset loaded successfully.")
    print("Shape of the dataset:", df.shape)
    print("\nFirst 5 rows of the dataset:")
    print(df.head())

    # --- Initial Data Exploration ---
    print("\nDataset Information:")
    df.info()

    # Check for missing values
    print("\nMissing values in each column:")
    print(df.isnull().sum())

    # Check the distribution of the target variable 'status'
    print("\nDistribution of the target variable 'status':")
    print(df['status'].value_counts())

    # Map the target variable to binary format (0 for legitimate, 1 for phishing)
    df['label'] = df['status'].map({'legitimate': 0, 'phishing': 1})

    # Drop the original 'status' column and the pre-extracted features, keeping only 'url' and our new 'label'
    # For this project, we will re-engineer features from the 'url' column.
    data = df[['url', 'label']].copy()

    print("\nCleaned dataset ready for feature extraction:")
    print(data.head())
    print("\nNew label distribution:")
    print(data['label'].value_counts())

except FileNotFoundError:
    print("Error: dataset_phishing.csv not found. Please ensure the Kaggle download was successful.")

In [ ]:
# Cell 3: URL-Based (Lexical) Feature Extraction

from urllib.parse import urlparse, urlsplit
import re
import ipaddress

# Function to check if the URL uses an IP address instead of a domain name
def has_ip_address(url):
    try:
        # Extract the netloc (domain part)
        netloc = urlparse(url).netloc
        # Check if the netloc is a valid IP address
        ipaddress.ip_address(netloc)
        return 1
    except ValueError:
        return 0

# Function to check for the presence of the '@' symbol
def has_at_symbol(url):
    return 1 if '@' in url else 0

# Function to determine URL length category
def get_url_length(url):
    # Phishers can use long URLs to hide the doubtful part in the address bar
    if len(url) < 54:
        return 0  # Legitimate
    elif 54 <= len(url) <= 75:
        return 0.5 # Suspicious
    else:
        return 1  # Phishing

# Function to calculate the depth of the URL (number of directories)
def get_url_depth(url):
    path = urlparse(url).path
    # Count the number of slashes in the path
    depth = path.count('/')
    return depth

# Function to check for redirection using '//'
def has_redirection(url):
    # The existence of “//” within the URL path means that the user will be redirected
    # We check for '//' after the protocol part (e.g., after https://)
    # A legitimate URL like "https://example.com" has its first '//' at position 6
    pos = url.find('//', 8) # Search after 'https://'
    return 1 if pos!= -1 else 0

# Function to check for "http" or "https" in the domain part
def has_http_in_domain(url):
    domain = urlparse(url).netloc
    return 1 if 'http' in domain or 'https' in domain else 0

# Function to check for a prefix or suffix separated by '-' in the domain
def has_hyphen_in_domain(url):
    domain = urlparse(url).netloc
    return 1 if '-' in domain else 0

# --- Additional Lexical Feature Functions ---

def count_dots(url):
    return url.count('.')

def count_hyphens(url):
    return url.count('-')

def count_underscores(url):
    return url.count('_')

def count_slashes(url):
    return url.count('/')

def count_question_marks(url):
    return url.count('?')

def count_equals(url):
    return url.count('=')

def count_www(url):
    return url.lower().count('www')

def count_digits(url):
    return sum(c.isdigit() for c in url)

def count_letters(url):
    return sum(c.isalpha() for c in url)

def get_hostname_length(url):
    return len(urlparse(url).netloc)

def get_path_length(url):
    return len(urlparse(url).path)

print("URL-based feature extraction functions defined.")

In [ ]:
# Cell 4 (Final Update): Domain-Based Feature Extraction with Global Timeout

import whois
from datetime import datetime
import tld
import time
import socket # Import socket to set a global timeout

# Set a global timeout for all socket operations (including those used by whois)
# This prevents any single request from hanging indefinitely.
socket.setdefaulttimeout(5) # 5 seconds

def get_domain_age(url):
    """
    Calculates the domain age in days, handling timeouts, timezone differences, and errors.
    """
    for i in range(3): # Try up to 3 times
        try:
            domain_name = tld.get_fld(url, fail_silently=True)
            if domain_name is None:
                return -1

            w = whois.whois(domain_name)

            creation_date = w.creation_date
            if isinstance(creation_date, list):
                creation_date = creation_date

            if creation_date:
                now = datetime.now()
                if creation_date.tzinfo:
                    creation_date = creation_date.replace(tzinfo=None)

                age = (now - creation_date).days
                return age
            else:
                return -1
        except Exception as e:
            # Catch any exception (including socket.timeout) and retry
            # print(f"Attempt {i+1} failed for {url} (Age): {e}. Retrying...")
            time.sleep(2 ** i) # Exponential backoff: 1, 2, 4 seconds
    return -1 # Return -1 if all retries fail

def get_domain_reg_len(url):
    """
    Calculates remaining registration length, handling timeouts, timezone differences, and errors.
    """
    for i in range(3): # Try up to 3 times
        try:
            domain_name = tld.get_fld(url, fail_silently=True)
            if domain_name is None:
                return -1

            w = whois.whois(domain_name)

            expiration_date = w.expiration_date
            if isinstance(expiration_date, list):
                expiration_date = expiration_date

            if expiration_date:
                now = datetime.now()
                if expiration_date.tzinfo:
                    expiration_date = expiration_date.replace(tzinfo=None)

                reg_len = (expiration_date - now).days
                return reg_len
            else:
                return -1
        except Exception as e:
            # Catch any exception (including socket.timeout) and retry
            # print(f"Attempt {i+1} failed for {url} (RegLen): {e}. Retrying...")
            time.sleep(2 ** i) # Exponential backoff
    return -1 # Return -1 if all retries fail

print("Domain-based feature extraction functions updated with global timeout and robust error handling.")

In [ ]:
# Cell 5: HTML and JavaScript-Based (Content) Feature Extraction

import requests
from bs4 import BeautifulSoup

# This function is more complex and involves network requests, which can be slow and fail.
# For a large-scale application, this should be handled with care (e.g., timeouts, asynchronous requests).

def extract_content_features(url):
    features = {
        'num_links': -1,
        'num_scripts': -1,
        'has_form': -1,
        'has_suspicious_form': -1,
        'has_iframe': -1,
        'external_link_ratio': -1
    }
    try:
        # Use a timeout to prevent hanging on unresponsive sites
        response = requests.get(url, timeout=5, verify=False, headers={'User-Agent': 'Mozilla/5.0'})

        # Check for successful response
        if response.status_code == 200:
            soup = BeautifulSoup(response.content, 'html.parser')

            # Number of links
            links = soup.find_all('a')
            features['num_links'] = len(links)

            # Number of scripts
            scripts = soup.find_all('script')
            features['num_scripts'] = len(scripts)

            # Presence of forms
            forms = soup.find_all('form')
            features['has_form'] = 1 if forms else 0

            # Presence of suspicious forms (e.g., blank action or submitting to a different domain)
            domain = urlparse(url).netloc
            features['has_suspicious_form'] = 0
            for form in forms:
                action = form.get('action', '').lower()
                if not action or 'mailto:' in action or (urlparse(action).netloc!= '' and urlparse(action).netloc!= domain):
                    features['has_suspicious_form'] = 1
                    break

            # Presence of iFrames
            features['has_iframe'] = 1 if soup.find_all('iframe') else 0

            # External link ratio
            if features['num_links'] > 0:
                external_links = 0
                for link in links:
                    href = link.get('href', '')
                    if href.startswith('http') and urlparse(href).netloc!= domain:
                        external_links += 1
                features['external_link_ratio'] = external_links / features['num_links']
            else:
                features['external_link_ratio'] = 0

    except requests.exceptions.RequestException as e:
        # If any request error occurs, we keep the default -1 values
        # print(f"Could not fetch content for {url}: {e}")
        pass

    return features

print("Content-based feature extraction function defined.")
# Example usage:
# content_feats = extract_content_features("https://www.google.com")
# print("Content features for google.com:", content_feats)

In [ ]:
# Cell 6: Master Feature Extraction and Dataset Creation

from tqdm.auto import tqdm

# Configure tqdm for pandas
tqdm.pandas()

def extract_all_features(url):
    features = {}

    # URL-Based Features
    features['has_ip_address'] = has_ip_address(url)
    features['has_at_symbol'] = has_at_symbol(url)
    features['url_length'] = get_url_length(url)
    features['url_depth'] = get_url_depth(url)
    features['has_redirection'] = has_redirection(url)
    features['has_http_in_domain'] = has_http_in_domain(url)
    features['has_hyphen_in_domain'] = has_hyphen_in_domain(url)
    features['count_dots'] = count_dots(url)
    features['count_hyphens'] = count_hyphens(url)
    features['count_underscores'] = count_underscores(url)
    features['count_slashes'] = count_slashes(url)
    features['count_question_marks'] = count_question_marks(url)
    features['count_equals'] = count_equals(url)
    features['count_www'] = count_www(url)
    features['count_digits'] = count_digits(url)
    features['count_letters'] = count_letters(url)
    features['hostname_length'] = get_hostname_length(url)
    features['path_length'] = get_path_length(url)

    # Domain-Based Features
    features['domain_age'] = get_domain_age(url)
    features['domain_reg_len'] = get_domain_reg_len(url)

    # Content-Based Features
    # Note: This is the slowest part. For a quick test, you can comment this section out.
    # content_feats = extract_content_features(url)
    # features.update(content_feats)

    return features

# --- Apply feature extraction to the dataset ---
# WARNING: This process can be very slow, especially with content features enabled.
# It may take over an hour to run on the full dataset in Colab.
# For demonstration purposes, we will run it on a smaller sample.

sample_data = data.sample(n=1000, random_state=42).copy() # Using a sample of 1000 for speed

print(f"Starting feature extraction for {len(sample_data)} URLs...")

# Use progress_apply to show a progress bar
feature_list = sample_data['url'].progress_apply(extract_all_features)

# Convert the list of dictionaries to a DataFrame
features_df = pd.DataFrame(feature_list.tolist(), index=sample_data.index)

# Combine features with the original labels
final_data = pd.concat([features_df, sample_data['label']], axis=1)

print("\nFeature extraction complete.")
print("Shape of the final dataset:", final_data.shape)
print("\nFirst 5 rows of the final dataset with extracted features:")
print(final_data.head())

In [ ]:
# Cell 7 (Updated): Data Cleaning, Splitting, and Scaling

# Handle missing values (impute with median)
# The value -1 was used as a placeholder for failed lookups.
for col in final_data.columns:
    if (final_data[col] == -1).any():
        median_val = final_data[col][final_data[col]!= -1].median()
        # --- MODIFIED LINE ---
        # Assign the result of the replace operation back to the column
        final_data[col] = final_data[col].replace(-1, median_val)

# Separate features (X) and target (y)
X = final_data.drop('label', axis=1)
y = final_data['label']

# Split the data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert scaled arrays back to DataFrames for easier handling
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)

print("Data preprocessing complete.")
print("Training set shape:", X_train_scaled.shape)
print("Testing set shape:", X_test_scaled.shape)

In [ ]:
# Cell 8: Feature Selection with SelectKBest

from sklearn.feature_selection import SelectKBest, chi2

# We need non-negative values for chi2, so we'll use the unscaled data for selection
# and apply the selection to the scaled data.
# Note: A scaler that preserves non-negativity (like MinMaxScaler) would be ideal for chi2.
# Here we proceed with a temporary non-negative version for demonstration.
X_train_non_negative = scaler.fit_transform(X_train)
min_val = X_train_non_negative.min()
X_train_non_negative -= min_val

# Select the top 15 features
k_best = 15
selector_kbest = SelectKBest(score_func=chi2, k=k_best)
X_train_kbest = selector_kbest.fit_transform(X_train_non_negative, y_train)

# Get the names of the selected features
selected_features_kbest_mask = selector_kbest.get_support()
selected_features_kbest = X_train.columns[selected_features_kbest_mask]

print(f"Top {k_best} features selected by SelectKBest:")
print(list(selected_features_kbest))

# Create new datasets with only the selected features
X_train_kbest_scaled = X_train_scaled[selected_features_kbest]
X_test_kbest_scaled = X_test_scaled[selected_features_kbest]

print("\nShape of data after SelectKBest:", X_train_kbest_scaled.shape)

In [ ]:
# Cell 8: Feature Selection with SelectKBest

from sklearn.feature_selection import SelectKBest, chi2

# We need non-negative values for chi2, so we'll use the unscaled data for selection
# and apply the selection to the scaled data.
# Note: A scaler that preserves non-negativity (like MinMaxScaler) would be ideal for chi2.
# Here we proceed with a temporary non-negative version for demonstration.
X_train_non_negative = scaler.fit_transform(X_train)
min_val = X_train_non_negative.min()
X_train_non_negative -= min_val

# Select the top 15 features
k_best = 15
selector_kbest = SelectKBest(score_func=chi2, k=k_best)
X_train_kbest = selector_kbest.fit_transform(X_train_non_negative, y_train)

# Get the names of the selected features
selected_features_kbest_mask = selector_kbest.get_support()
selected_features_kbest = X_train.columns[selected_features_kbest_mask]

print(f"Top {k_best} features selected by SelectKBest:")
print(list(selected_features_kbest))

# Create new datasets with only the selected features
X_train_kbest_scaled = X_train_scaled[selected_features_kbest]
X_test_kbest_scaled = X_test_scaled[selected_features_kbest]

print("\nShape of data after SelectKBest:", X_train_kbest_scaled.shape)

In [ ]:
# Cell 9: Feature Selection with Recursive Feature Elimination (RFE)

from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier

# Initialize the base estimator
estimator = RandomForestClassifier(n_estimators=50, random_state=42)

# Initialize RFE
# We will select the same number of features for a fair comparison
k_rfe = 15
selector_rfe = RFE(estimator=estimator, n_features_to_select=k_rfe, step=1)

# Fit RFE on the scaled training data
selector_rfe = selector_rfe.fit(X_train_scaled, y_train)

# Get the names of the selected features
selected_features_rfe_mask = selector_rfe.get_support()
selected_features_rfe = X_train.columns[selected_features_rfe_mask]

print(f"Top {k_rfe} features selected by RFE:")
print(list(selected_features_rfe))

# Create new datasets with only the RFE-selected features
# We will use this set for the remainder of the analysis
X_train_selected = X_train_scaled[selected_features_rfe]
X_test_selected = X_test_scaled[selected_features_rfe]

print("\nShape of data after RFE:", X_train_selected.shape)

In [ ]:
# Cell 10: Model Initialization

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Initialize the models with standard parameters
models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "Naive Bayes": GaussianNB(),
    "Support Vector Machine": SVC(probability=True, random_state=42), # probability=True for ROC-AUC
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

print("Models initialized.")

In [ ]:
# Cell 11: Model Training

# Dictionary to store the trained models
trained_models = {}

print("Starting model training...")

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train_selected, y_train)
    trained_models[name] = model
    print(f"{name} trained.")

print("\nAll models have been trained successfully.")

In [ ]:
# Cell 12: Model Evaluation on Test Set

# Dictionary to store evaluation results
results = {}

print("Evaluating models on the test set...")

for name, model in trained_models.items():
    # Make predictions on the test data
    y_pred = model.predict(X_test_selected)
    y_proba = model.predict_proba(X_test_selected)[:, 1] # Probabilities for the positive class

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    # Store results
    results[name] = {
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'AUC-ROC': auc
    }
    print(f"--- {name} ---")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")
    print(f"AUC-ROC: {auc:.4f}\n")

# Convert results to a DataFrame for better visualization
results_df = pd.DataFrame(results).T

In [ ]:
# Cell 13 (Updated): K-Fold Cross-Validation

from sklearn.model_selection import cross_val_score, StratifiedKFold
import pandas as pd

# Set up 10-fold stratified cross-validation
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

cv_scores = {}

print("Performing 10-Fold Cross-Validation...")

# --- MODIFICATION HERE ---
# First, scale the entire dataset 'X' using the fitted scaler.
# The scaler expects the same columns it was fitted on (from X_train).
X_full_scaled = scaler.transform(X)

# Convert the scaled numpy array back to a DataFrame to select columns by name.
X_full_scaled_df = pd.DataFrame(X_full_scaled, columns=X.columns)

# Now, select the features chosen by RFE from the fully scaled data.
X_selected_for_cv = X_full_scaled_df[selected_features_rfe]
# --- END MODIFICATION ---

for name, model in models.items(): # Use the un-trained models for CV
    print(f"Cross-validating {name}...")
    # Use the correctly prepared dataset for cross-validation
    scores = cross_val_score(model, X_selected_for_cv, y, cv=cv, scoring='accuracy', n_jobs=-1)
    cv_scores[name] = scores.mean()
    print(f"{name} - Mean CV Accuracy: {scores.mean():.4f} (+/- {scores.std():.4f})")

# Add CV scores to the results DataFrame
results_df['Mean CV Accuracy'] = pd.Series(cv_scores)

In [ ]:
# Cell 14: Display Comparative Results Table

print("--- Comparative Performance of Classification Models ---")
# Sort by F1-Score as it balances Precision and Recall
results_df_sorted = results_df.sort_values(by='F1-Score', ascending=False)
print(results_df_sorted)

# Plotting the results for visual comparison
results_df_sorted.plot(kind='bar', figsize=(14, 8))
plt.title('Model Performance Comparison')
plt.ylabel('Score')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--')
plt.legend(loc='lower right')
plt.show()

In [ ]:
# Cell 15 (Corrected): Plotting AUC-ROC Curves

plt.figure(figsize=(12, 10))

for name, model in trained_models.items():
    # Get prediction probabilities
    y_proba = model.predict_proba(X_test_selected)[:, 1]

    # Calculate ROC curve
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)

    # Plot ROC curve
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.4f})')

# Plot the random guess line from (0, 0) to (1, 1)
plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')
# --- END MODIFICATION ---

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curves')
plt.legend(loc='lower right')
plt.grid()
plt.show()

In [ ]:
# Cell 16: Model Persistence (Saving the Best Model and Scaler)

import joblib

# Let's assume Random Forest is our chosen model based on a good balance of performance metrics
best_model_name = 'Random Forest'
best_model = trained_models[best_model_name]

# Save the model
model_filename = 'phishing_model.joblib'
joblib.dump(best_model, model_filename)
print(f"Model '{best_model_name}' saved to {model_filename}")

# Save the scaler
scaler_filename = 'scaler.joblib'
joblib.dump(scaler, scaler_filename)
print(f"Scaler saved to {scaler_filename}")

# Save the list of selected features
features_filename = 'selected_features.joblib'
joblib.dump(selected_features_rfe, features_filename)
print(f"Selected features list saved to {features_filename}")

In [ ]:
# Cell 17: Creating the Full Prediction Pipeline

def predict_url(url):
    """
    Takes a raw URL string, performs full feature extraction,
    loads the saved model and scaler, and returns a prediction.
    Returns prediction (string), and probability (numpy array or None)
    """
    try:
        loaded_model = joblib.load('phishing_model.joblib')
        loaded_scaler = joblib.load('scaler.joblib')
        loaded_features_list = joblib.load('selected_features.joblib')
    except FileNotFoundError:
        return "Error: Model or scaler files not found. Please run the training and saving steps first.", None

    # 1. Feature Extraction
    features_dict = extract_all_features(url)

    # Check if feature extraction was successful (basic check)
    if not features_dict:
        return "Error: Could not extract features for this URL.", None

    features_df = pd.DataFrame([features_dict])

    # 2. Preprocessing and Cleaning Extracted Features
    # Handle potential missing values (-1) from extraction failures.
    # It's important to replace these with a value the model can handle,
    # ideally the median/mean from the training data, but for simplicity
    # and demonstration, we'll use 0 or a common fill value.
    # A more robust approach would load the median/mean from training.
    for col in features_df.columns:
        # Ensure the column exists and replace -1
        if col in features_df.columns and (features_df[col] == -1).any():
             # Replace -1 with a suitable fill value, e.g., the median from the training data
             # For now, using 0 or the median from the current small sample is a workaround.
             # A truly production-ready system would save/load these imputation values.
            if col in X.columns: # Check if the column was in the original training features
                 # Attempt to use the median from the training data if available
                 try:
                     training_median = X[col][X[col] != -1].median() # Calculate median from original training data (if it exists)
                     features_df[col] = features_df[col].replace(-1, training_median)
                 except Exception:
                     # Fallback to 0 if median calculation fails (e.g., all values are -1)
                     features_df[col] = features_df[col].replace(-1, 0)
            else:
                 # If the column was not in the original training data (unexpected), replace with 0
                 features_df[col] = features_df[col].replace(-1, 0)

        # Ensure all values are numerical after handling -1
        if not pd.api.types.is_numeric_dtype(features_df[col]):
             features_df[col] = pd.to_numeric(features_df[col], errors='coerce').fillna(0) # Coerce non-numeric to NaN, then fill NaN with 0


    # Ensure the order and presence of columns match the scaler's expected input
    # This is critical. If feature extraction produced different columns, scaling will fail.
    # A robust solution would save and load the list of columns from the training data.
    if not loaded_scaler.feature_names_in_ is None:
        expected_cols = loaded_scaler.feature_names_in_
        # Reindex features_df to match the expected columns, filling missing with 0
        features_df = features_df.reindex(columns=expected_cols, fill_value=0)
    elif set(features_df.columns) != set(X.columns): # Fallback if scaler has no feature_names_in_
         print("Warning: Column mismatch between extracted features and training data.")
         # Attempt to align columns based on training data columns (X.columns)
         try:
             features_df = features_df.reindex(columns=X.columns, fill_value=0)
         except Exception as e:
             return f"Error: Could not align extracted features with training columns: {e}", None


    # 3. Scaling
    try:
        features_scaled = loaded_scaler.transform(features_df)
        features_scaled_df = pd.DataFrame(features_scaled, columns=features_df.columns)
    except Exception as e:
        return f"Error during scaling: {e}", None


    # 4. Feature Selection
    # Ensure the columns in features_scaled_df match the columns in loaded_features_list
    try:
        final_features = features_scaled_df[loaded_features_list]
    except KeyError as e:
         return f"Error: Feature mismatch during selection. Missing feature: {e}", None


    # 5. Prediction
    try:
        prediction = loaded_model.predict(final_features)
        probability = loaded_model.predict_proba(final_features)
    except Exception as e:
        return f"Error during prediction: {e}", None


    # 6. Return result
    if prediction == 1:
        result = "Phishing"
    else:
        result = "Legitimate"

    # probability is typically a 2D array like [[prob_class_0, prob_class_1]]
    return result, probability

print("Prediction pipeline function 'predict_url' updated with more robust error handling.")

In [ ]:
# Cell 18: Live Demonstration of the Prediction Pipeline

# Example 1: A known legitimate URL
legit_url = "https://www.github.com/scikit-learn/scikit-learn"
result, confidence = predict_url(legit_url)
print(f"URL: {legit_url}")
print(f"Prediction: {result}")
# Access the probability of the predicted class
predicted_class_index = 0 if result == "Legitimate" else 1
print(f"Confidence: {confidence[0][predicted_class_index]:.2%}\n")


# Example 2: A suspicious URL (hypothetical example with phishing characteristics)
# Characteristics: IP address, long path, suspicious keywords
phishing_url = "http://192.168.1.1/paypal.com-secure-login/update-your-account-details.html?user=victim"
result, confidence = predict_url(phishing_url)
print(f"URL: {phishing_url}")
print(f"Prediction: {result}")
# Access the probability of the predicted class
predicted_class_index = 0 if result == "Legitimate" else 1
print(f"Confidence: {confidence[0][predicted_class_index]:.2%}\n")

# Example 3: Another legitimate URL
legit_url_2 = "https://www.youtube.com/watch?v=dQw4w9WgXcQ"
result, confidence = predict_url(legit_url_2)
print(f"URL: {legit_url_2}")
print(f"Prediction: {result}")
# Access the probability of the predicted class
predicted_class_index = 0 if result == "Legitimate" else 1
print(f"Confidence: {confidence[0][predicted_class_index]:.2%}\n")

In [ ]:
# Cell 19: Confusion Matrix for the Best Model

from sklearn.metrics import ConfusionMatrixDisplay

# Identify the best model from our results (e.g., Random Forest)
# --- MODIFIED LINE ---
# Specify the column to find the max index for
best_model_name = results_df['F1-Score'].idxmax()
best_model = trained_models[best_model_name]

print(f"--- In-Depth Analysis for: {best_model_name} ---")

# Plot the confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
y_pred = best_model.predict(X_test_selected)
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax, display_labels=['Legitimate', 'Phishing'], cmap=plt.cm.Blues)
ax.set_title(f'Confusion Matrix for {best_model_name}')
plt.show()

In [ ]:
# Cell 20: ROC Curve for the Best Model

from sklearn.metrics import RocCurveDisplay

# Plot the ROC curve for the best model
fig, ax = plt.subplots(figsize=(8, 6))
RocCurveDisplay.from_estimator(best_model, X_test_selected, y_test, ax=ax, name=best_model_name)
ax.plot([0, 1], [0, 1], 'k--', label='Random Guess') # Add the random guess line
ax.set_title(f'ROC Curve for {best_model_name}')
ax.legend()
plt.show()

In [ ]:
# Cell 21: Feature Importance Plot for the Best Model

# Extract feature importances
importances = best_model.feature_importances_
feature_names = X_train_selected.columns

# Create a pandas series for easier plotting
feature_importance_series = pd.Series(importances, index=feature_names).sort_values(ascending=True)

# Plot the feature importances
plt.figure(figsize=(10, 8))
feature_importance_series.plot(kind='barh', color='teal')
plt.title(f'Feature Importance for {best_model_name}')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 22: New Function to Display Features and Prediction

import pandas as pd
import numpy as np # Import numpy to check array types

def analyze_and_predict_url(url):
    """
    Takes a raw URL, displays its extracted features, and then shows the
    model's prediction and confidence.
    """
    print("--- Feature Extraction ---")
    # Extract features from the URL
    features_dict = extract_all_features(url)

    if not features_dict:
        print("Error: Could not extract features for this URL.")
        return

    # Convert dictionary to a DataFrame for clean printing
    features_df = pd.DataFrame([features_dict]).T
    features_df.columns = ['Value']

    # Display the extracted features
    print(features_df)
    print("\n" + "="*30 + "\n")

    print("--- Model Prediction ---")
    # Get the prediction from the existing pipeline
    result, probability = predict_url(url)

    # Display the result
    print(f"URL: {url}")
    print(f"Prediction: {result}")

    # Check if probability is a valid numpy array with shape (1, 2)
    if isinstance(probability, np.ndarray) and probability.shape == (1, 2):
        print(f"Confidence -> Legitimate: {probability[0][0]:.2%}, Phishing: {probability[0][1]:.2%}")
    elif isinstance(result, str) and result.startswith("Error:"):
        # If predict_url returned an error string, display it
        print(f"Prediction Pipeline Error: {result}")
        print("Confidence: Not available due to pipeline error")
    else:
        # Handle other unexpected return types from predict_url
        print(f"Confidence: Not available (Unexpected output from predict_url. Result type: {type(result)}, Probability type: {type(probability)})")


print("Function 'analyze_and_predict_url' is updated for more robust output handling.")

In [ ]:
# Cell 23.1: Demonstration of the New Analysis Function

# You can test any URL here
test_url = "https://www.wikipedia.org"

analyze_and_predict_url(test_url)

In [ ]:
# Cell 23.2: Demonstration of the New Analysis Function

# You can test any URL here
test_url = "https://www.cnn.com/specials/us/energy-and-environment"

analyze_and_predict_url(test_url)

In [ ]:
# Cell 23.3: Demonstration of the New Analysis Function

# You can test any URL here
test_url = "http://microsoft.com@123.45.67.89/secure-login"

analyze_and_predict_url(test_url)

In [ ]:
# Cell 23.4: Demonstration of the New Analysis Function

# You can test any URL here
test_url = "http://192.168.1.1/login.html"

analyze_and_predict_url(test_url)

In [ ]:
# Cell 23.4: Demonstration of the New Analysis Function

# You can test any URL here
test_url = "http://signin.google.com-secure.net/"

analyze_and_predict_url(test_url)

In [ ]:
# Cell 23.4: Demonstration of the New Analysis Function

# You can test any URL here
test_url = " https://www.amazon-support.co/"

analyze_and_predict_url(test_url)

Legitimate URLs for Testing

https://www.wikipedia.org

https://www.cnn.com/specials/us/energy-and-environment

https://github.com/scikit-learn/scikit-learn

Phishing & Suspicious URLs for Testing

http://192.168.1.1/login.html

http://signin.google.com-secure.net/

https://www.amazon-support.co/

http://microsoft.com@123.45.67.89/secure-login
